# 08 — Short interest, lending and investor flow

**The question:** *Which Brazilian names are most heavily shorted, how expensive
is the borrow, who is lending it — and who was buying while that was happening?*

Four views, all fed by B3's daily BDI files:

| View | One row per | Answers |
| --- | --- | --- |
| `short_interest` | `(ticker, trade_date)` | how big the short book is, how expensive, how long to cover |
| `short_interest_by_sector` | `(sector, trade_date)` | the same book by B3 top-level sector |
| `lending_trades` | `(ticker, trade_date)` | the lending tape: trades, quantity, rate min/mean/max |
| `lending_participants` | `(ticker, trade_date, broker)` | which brokerage was on each leg |
| `investor_flow` | `(investor_type, reference_date)` | who was buying and selling |

**The key column is `ticker`, not `codneg`.**

<hr/>

### Two things to know before the first call

**1. This is the one part of the warehouse that is a ratchet.** B3 keeps roughly
**21 business days** of these files and publishes no archive. A session the
daily job misses is lost at any price — nothing can backfill it. So the history
below is exactly as deep as the pipeline has been running, and no deeper.

**2. These five views are live but not yet on the docs site**, and not yet
listed in `catalog().postgrest` (v26). The SDK's `view()` allow-list therefore
rejects them, so this notebook talks to PostgREST directly. When first-class SDK
methods land, replace `rest(...)` below with `silo.short_interest(...)` — the
rows are the same.

In [ ]:
# The SDK is not on PyPI. From the repository root:
#
#     pip install -e sdk/
#
# Auth is the shared publishable key printed in the docs. It is for TESTING:
# everyone reading the docs has the same one, so it identifies the project and
# not you. It puts you on the ANONYMOUS tier. Set SILO_TOKEN to a GitHub
# sign-in token (notebook 00) to run signed in.
import os

os.environ.setdefault("SILO_URL", "https://zcjbtpxuhdekpwcxmepn.supabase.co")
os.environ.setdefault(
    "SILO_ANON_KEY", "sb_publishable__yfFQsykAglrvc9GS6_PYw_B24ex437"
)

import pandas as pd

from silo_client import SiloClient

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

silo = SiloClient()
print(f"tier            : {silo.tier}")
print(f"catalog version : {silo.catalog()['version']}")

In [ ]:
def as_of(*datasets: str) -> pd.DataFrame:
    """Print how fresh every dataset this notebook relies on actually is.

    Run this FIRST, every time. A stale warehouse then shows up in the output
    instead of being silently baked into a number further down.

      as_of            the newest period that has landed AND has elapsed.
                       This is freshness.
      complete_through the newest period classified COMPLETE — what the
                       default windows serve.
      newest_period    the newest period KEY. It can sit in the FUTURE when a
                       family files forward-dated (FIP is keyed 31-December).
                       Never read this as freshness.
      landed_at        when ingest last SUCCEEDED. A later failed run never
                       advances it.
      notes            a caveat the dates cannot carry. Printed in full below,
                       never summarised, never dropped.
    """
    cov = pd.DataFrame(silo.coverage())
    rows = cov[cov["dataset"].isin(datasets)].copy()
    missing = set(datasets) - set(rows["dataset"])
    if missing:
        raise RuntimeError(f"coverage() has no row for {sorted(missing)}")
    print(
        rows[
            ["dataset", "as_of", "complete_through", "newest_period", "landed_at"]
        ].to_string(index=False)
    )
    for r in rows.itertuples():
        if r.notes:
            print(f"\n  CAVEAT [{r.dataset}]\n  {r.notes}")
    return rows.set_index("dataset")

In [ ]:
import httpx

REST = f"{os.environ['SILO_URL'].rstrip('/')}/rest/v1"
HEADERS = {"apikey": os.environ["SILO_ANON_KEY"], "Prefer": "count=exact"}


def rest(view: str, **filters) -> pd.DataFrame:
    """One page of a PostgREST view, with the truncation guard the SDK gives you.

    TEMPORARY. These five views are not in SiloClient.VIEWS yet, so
    silo.view() refuses them by name. Everything else here — the filter
    syntax, the 1000-row cap, Content-Range as the only signal — is identical
    to any other view.
    """
    r = httpx.get(f"{REST}/{view}", params=filters, headers=HEADERS, timeout=30)
    if r.status_code >= 400:
        raise RuntimeError(f"HTTP {r.status_code} from {view}: {r.text[:300]}")
    rows = r.json()
    total = (r.headers.get("Content-Range") or "").rsplit("/", 1)[-1]
    rest.last_total = int(total) if total.isdigit() else None
    if "limit" not in filters and total.isdigit() and int(total) > len(rows):
        # Exactly the SiloTruncated case: a view was cut and answered 200.
        raise RuntimeError(
            f"{view} returned {len(rows)} of {total} rows and was CUT at the "
            f"server's 1000-row cap. Filter it or pass an explicit limit."
        )
    return pd.DataFrame(rows)

## Freshness, when `coverage()` has no row for you

`coverage()` does not carry these datasets yet, so the honest thing is to read
the window off the views themselves and print it. Same discipline, different
source: a stale warehouse has to be visible in the output.

In [ ]:
COVERAGE = as_of("quotes")     # the ADTV denominator comes from the cash tape
print()

windows = {}
for view, datecol in (("short_interest", "trade_date"),
                      ("short_interest_by_sector", "trade_date"),
                      ("lending_trades", "trade_date"),
                      ("lending_participants", "trade_date"),
                      ("investor_flow", "reference_date")):
    newest = rest(view, select=datecol, order=f"{datecol}.desc", limit=1)
    oldest = rest(view, select=datecol, order=f"{datecol}.asc", limit=1)
    windows[view] = (oldest.iloc[0, 0], newest.iloc[0, 0])
    print(f"  {view:26s} {windows[view][0]} .. {windows[view][1]}")

print()
print("CAVEAT: B3 retains ~21 business days of these files and keeps NO archive.")
print("        The oldest date above is when this pipeline started capturing,")
print("        not when the market started lending. Nothing can backfill it.")

In [ ]:
SESSION = windows["short_interest"][1]
print(f"working on session {SESSION}")

## The short book

`short_value` is the open short balance in BRL. Two derived columns beside it
carry caveats that are **published as columns** rather than left to you.

In [ ]:
si = rest("short_interest", trade_date=f"eq.{SESSION}",
          order="short_value.desc", limit=15)

si[["ticker", "asset_name", "instrument_category", "short_value",
    "pct_float", "float_basis", "days_to_cover",
    "lender_rate_pct", "borrower_rate_pct"]]

### `pct_float` is **two different metrics**, and `float_basis` says which

B3 publishes a real free float only for **index constituents** (~149 tickers).
For everything else the honest denominator is **shares outstanding**, which is a
**larger** number and therefore yields a **smaller** percentage.

They are not the same metric. Ranking across the two produces a league table
where the basis, not the short interest, decides the order.

In [ ]:
day = rest("short_interest", trade_date=f"eq.{SESSION}",
           select="ticker,short_value,pct_float,float_basis,days_to_cover,"
                  "adtv_value_21,adtv_sessions",
           order="short_value.desc", limit=1000)

print(f"{len(day):,} rows received; {rest.last_total:,} tickers had a short "
      f"balance on {SESSION}.")
if rest.last_total > len(day):
    print("This is the top page by short_value, not the whole book — 1,000 rows")
    print("is PostgREST's server-wide page and no tier raises it. Everything")
    print("below describes that page and says so.")
print()
print(day["float_basis"].value_counts(dropna=False).to_frame("tickers").to_string())
print()
print("median pct_float BY BASIS — the gap is the basis, not the shorting:\n")
print(day.groupby("float_basis", dropna=False)["pct_float"]
      .agg(["count", "median", "max"]).to_string(float_format=lambda v: f"{v:,.3f}"))

In [ ]:
top = day.head(12).copy()
top["comparable_with"] = top["float_basis"]
print(f"Top short balances, {SESSION} — NOT a single ranking:\n")
print(top[["ticker", "short_value", "pct_float", "float_basis"]]
      .to_string(index=False, float_format=lambda v: f"{v:,.3f}"))
print()
print("CAVEAT: pct_float is comparable only WITHIN one float_basis.")
print("        index_free_float   = B3's published free float (index members)")
print("        shares_outstanding = total capital — a LARGER denominator, so a")
print("                             SMALLER percentage for the same short book")

In [ ]:
# The right way to rank: one basis at a time.
for basis, grp in day.dropna(subset=["float_basis"]).groupby("float_basis"):
    print(f"\nmost shorted on basis {basis!r}:")
    print(grp.nlargest(6, "pct_float")[["ticker", "pct_float", "short_value"]]
          .to_string(index=False, float_format=lambda v: f"{v:,.2f}"))

### `days_to_cover` is `null`, never `0`, when it cannot be computed

`days_to_cover` divides the short balance by a trailing 21-session average daily
traded value. When a ticker has not traded in the window — or the trailing
average is zero — the ratio is **`null`**.

It is deliberately not `0`. A short position in an untraded name is
**uncoverable**, not instantly coverable, and `0` would sort it to exactly the
wrong end of a screen.

In [ ]:
no_dtc = day[day["days_to_cover"].isna()]
print(f"tickers with a short balance but NO days_to_cover: {len(no_dtc)} "
      f"of {len(day)}")
print()
if len(no_dtc):
    print(no_dtc[["ticker", "short_value", "adtv_value_21", "adtv_sessions"]]
          .head(8).to_string(index=False, float_format=lambda v: f"{v:,.2f}"))
    print()
print("A null here means UNCOVERABLE, not 'covers instantly'. Dropping these")
print("rows is a choice; filling them with 0 is a fabrication.")

### By sector

In [ ]:
sector = rest("short_interest_by_sector", trade_date=f"eq.{SESSION}",
              order="short_value.desc", limit=30)
print(sector.to_string(index=False, float_format=lambda v: f"{v:,.2f}"))
print()
print("CAVEAT: sector is B3's published classification, and it is NULL for")
print("        instruments that have none (ETFs, BDRs). Those tickers are in")
print("        short_interest but not under any sector row here, so the sector")
print("        rows do NOT sum to the whole book.")

## The lending tape

`lending_trades` is one row per `(ticker, trade_date)` over the trade-by-trade
file: how many trades, how much stock, and the rate — mean, min and max.

In [ ]:
TICKER = "PETR4"

tape = rest("lending_trades", ticker=f"eq.{TICKER}",
            order="trade_date.desc", limit=30)
tape[["trade_date", "trades", "quantity", "rate_pct", "rate_min_pct",
      "rate_max_pct", "borrower_brokers", "lender_brokers", "internal_trades"]]

Note the spread between `rate_min_pct` and `rate_max_pct` within a single
session. The borrow is not one price — `rate_pct` is a **mean**, and quoting it
alone hides a book that traded from one end of the range to the other.

In [ ]:
if not tape.empty:
    t = tape.copy()
    t["spread"] = t["rate_max_pct"] - t["rate_min_pct"]
    t["internal_pct"] = (100 * t["internal_trades"] / t["trades"]).round(1)
    print(f"{TICKER} borrow rate, most recent sessions:\n")
    print(t[["trade_date", "rate_min_pct", "rate_pct", "rate_max_pct",
             "spread", "trades", "internal_trades", "internal_pct"]]
          .head(10).to_string(index=False))

## `doador` / `tomador` are **brokerages**, not beneficial owners

This is the caveat that matters most on `lending_participants`, and it is not
subtle: **roughly 75 % of lending trades carry the same broker code on both
legs.** A large borrow through a broker is that broker's **client book**, not the
broker's own position.

`internal_legs` / `internal_qty` measure exactly that — how much of the flow
never left the house.

In [ ]:
parts = rest("lending_participants", ticker=f"eq.{TICKER}",
             trade_date=f"eq.{SESSION}", order="quantity_net.desc", limit=100)

if parts.empty:
    print(f"no lending legs for {TICKER} on {SESSION}")
else:
    print(f"{len(parts)} brokers on the {TICKER} book, {SESSION}\n")
    print(parts[["broker_code", "broker_name", "quantity_lent",
                 "quantity_borrowed", "quantity_net", "lender_rate_pct",
                 "borrower_rate_pct", "internal_legs"]]
          .head(10).to_string(index=False, float_format=lambda v: f"{v:,.0f}"))

In [ ]:
if not parts.empty:
    both = ((parts["quantity_lent"] > 0) & (parts["quantity_borrowed"] > 0)).sum()
    print(f"brokers on the book              : {len(parts)}")
    print(f"brokers appearing on BOTH legs   : {both} "
          f"({100 * both / len(parts):.0f}%)")
    print(f"internal legs (both legs same broker), summed: "
          f"{int(parts['internal_legs'].fillna(0).sum()):,}")
    print(f"internal quantity                : "
          f"{parts['internal_qty'].fillna(0).sum():,.0f}")
    print()
    print("Whoever sits at the top of this table is the BROKERAGE that cleared")
    print("the trade. It is not a fund, not a beneficial owner, and naming it")
    print("as 'the short seller' would be an invention. No beneficial-owner")
    print("identity exists anywhere in this data.")

## Who was buying: `investor_flow`

B3 publishes a **month-to-date cumulative** snapshot by investor type. The daily
figure here is its **first difference within a month** — derived, and lagging
about T+2.

`flow_basis` says how each row was produced, and it is the column to read
first:

| `flow_basis` | Means |
| --- | --- |
| `delta` | a genuine day-over-day difference |
| `month_open` | the month's first session — the MTD figure *is* the day |
| `unknown_opening_snapshot` | the first snapshot held in a month, when that was **not** the month's first session. Flows are **`null` on purpose.** |

That last row type is the interesting one. Reporting a part-month cumulative
total as a single day's flow would invent a spike, so the flows are left null
instead.

In [ ]:
flow = rest("investor_flow", order="reference_date.asc", limit=1000)
flow["reference_date"] = pd.to_datetime(flow["reference_date"])

print(flow["flow_basis"].value_counts().to_frame("rows").to_string())
print()
print("rows with NULL net flow, by basis:")
print(flow.assign(is_null=flow["net_value_thousands"].isna())
      .groupby("flow_basis")["is_null"].sum().to_frame("null_net").to_string())

In [ ]:
opening = flow[flow["flow_basis"] == "unknown_opening_snapshot"]
if not opening.empty:
    print("The deliberately-null rows:\n")
    print(opening[["reference_date", "investor_type", "flow_basis",
                   "buy_value_thousands", "net_value_thousands",
                   "mtd_buy_value_thousands"]]
          .to_string(index=False))
    print()
    print("mtd_* is populated; the daily columns are not. The cumulative total")
    print("is real, the implied 'day' is not, so only the real one is served.")

In [ ]:
usable = flow[flow["flow_basis"] == "delta"]
pivot = usable.pivot_table(index="reference_date", columns="investor_type",
                           values="net_value_thousands", aggfunc="sum")

print("net flow by investor type, R$ THOUSANDS, delta rows only\n")
print(pivot.tail(8).to_string(float_format=lambda v: f"{v:,.0f}"))
print()
print(f"CAVEAT: unit is R$ thousands, as B3 publishes it.")
print(f"CAVEAT: derived from month-to-date snapshots and lagging about T+2 —")
print(f"        the newest row here is {flow['reference_date'].max():%Y-%m-%d}, "
      f"behind the quote tape's {COVERAGE.loc['quotes', 'as_of']}.")

In [ ]:
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(11, 4))
for col in pivot.columns:
    ax.plot(pivot.index, pivot[col] / 1e6, marker="o", markersize=2.5, label=col)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_ylabel("net flow (R$ billions)")
ax.set_title(f"B3 net flow by investor type — delta rows only, "
             f"{pivot.index.min():%Y-%m-%d} .. {pivot.index.max():%Y-%m-%d}")
ax.legend(fontsize=7, ncol=3)
ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

print("Sessions whose flow_basis was not 'delta' are ABSENT from this chart,")
print("not zero and not interpolated. The line breaks where the data does.")

## Putting it together, carefully

A fair question: *did the most-shorted names see foreign selling?* The honest
answer is that **this data cannot tell you** — `investor_flow` is a
market-wide aggregate by investor type, with no ticker on it, and
`short_interest` has no investor type. There is no join, and inventing one by
lining up dates would be exactly the fabrication these notebooks exist to avoid.

What you *can* do is put them side by side and say so.

In [ ]:
si_hist = rest("short_interest", ticker=f"eq.{TICKER}",
               select="trade_date,short_value,pct_float,float_basis,days_to_cover",
               order="trade_date.asc", limit=100)
si_hist["trade_date"] = pd.to_datetime(si_hist["trade_date"])

basis = si_hist["float_basis"].dropna().unique()
print(f"{TICKER}: {len(si_hist)} sessions, "
      f"{si_hist['trade_date'].min():%Y-%m-%d} .. "
      f"{si_hist['trade_date'].max():%Y-%m-%d}")
print(f"float_basis over the window: {list(basis)}")
if len(basis) > 1:
    print("  ^ the basis CHANGED mid-window. pct_float is not a continuous")
    print("    series across that change — read short_value instead.")
print()
print(si_hist.set_index("trade_date").tail(10)
      .to_string(float_format=lambda v: f"{v:,.3f}"))

In [ ]:
foreign = pivot.get("Investidor Estrangeiro")
if foreign is not None:
    side_by_side = pd.DataFrame({
        f"{TICKER}_short_value": si_hist.set_index("trade_date")["short_value"],
        "foreign_net_flow_thousands": foreign,
    })
    print("SIDE BY SIDE — two different populations, deliberately not joined:\n")
    print(side_by_side.tail(10).to_string(float_format=lambda v: f"{v:,.0f}"))
    print()
    print("Note the NaNs where one series has a session the other does not.")
    print("They are left as NaN. Aligning them by filling would manufacture")
    print("agreement between two datasets that do not share a grain.")

## What this notebook did not claim

* It did not rank `pct_float` across two `float_basis` values.
* It did not fill a `days_to_cover` null with `0`.
* It did not treat an `unknown_opening_snapshot` row as a day's flow.
* It did not call a brokerage a short seller.
* It did not join `investor_flow` to a ticker, because no such join exists.
* It did not present the retention window as market history.

## Where this goes next

* Notebook `01` is the cash tape the ADTV denominator comes from.
* Notebook `04` is credit stress from the filing side rather than the market
  side.
* When the SDK gains first-class methods for these five views and `catalog()`
  lists them, `rest(...)` above becomes `silo.short_interest(...)` and the
  `rest` helper can be deleted. Nothing else in this notebook changes.

---

## The rules this notebook obeyed

* **Nothing was filled.** No forward-fill, no interpolation, no carried-forward
  last observation. A gap in a chart is a gap in the filings.
* **Every caveat was printed beside its number** — `coverage().notes`,
  `catalog().regime_breaks`, `catalog().applicability`, `float_basis` — rather
  than left in a docstring somewhere.
* **Freshness came from `coverage()`**, called before anything was claimed.

The contract these rules come from is
[Conventions & limits](https://octo-98895abd.mintlify.site/api-docs/conventions),
and its machine-readable twin is `POST /rpc/catalog`.